# Skills Ontology Mapper - Exploration & Training

This notebook explores the dataset, evaluates fuzzy matching techniques, and trains the TF-IDF vectorizer for semantic similarity.

In [ ]:
import json
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load ontology
with open('../data/ontology.json', 'r', encoding='utf-8') as f:
    ontology = json.load(f)

print(f"Loaded {len(ontology)} canonical skills.")

## 1. Fuzzy Matching Exploration
Let's test how well standard sequence ratios work on typos.

In [ ]:
from difflib import SequenceMatcher

test_cases = [
    ("pytohn", "python"),
    ("java script", "javascript"),
    ("react js", "react")
]

for raw, target in test_cases:
    ratio = SequenceMatcher(None, raw, target).ratio()
    print(f"'{raw}' vs '{target}': {ratio:.2f}")

## 2. TF-IDF Training
We train a character n-gram TF-IDF vectorizer on the display names and synonyms. This allows us to handle missing context and extreme typos.

In [ ]:
docs = []
skills = []
for skill in ontology:
    doc = skill["display_name"] + " " + " ".join(skill.get("synonyms", []))
    docs.append(doc.lower())
    skills.append(skill)

vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=5000)
tfidf_matrix = vectorizer.fit_transform(docs)

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

## 3. Test TF-IDF Semantic Search

In [ ]:
queries = ["Sr. Java Developer", "ML Expert", "pytohn"]
for q in queries:
    q_vec = vectorizer.transform([q.lower()])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    best_idx = scores.argmax()
    print(f"Query: '{q}' -> Best Match: {skills[best_idx]['display_name']} (Score: {scores[best_idx]:.2f})")

## 4. Persist the Model
Saving the trained vectorizer and matrix for inference.

In [ ]:
import pickle

model_artifacts = {
    "vectorizer": vectorizer,
    "tfidf_matrix": tfidf_matrix,
    "tfidf_skills": skills
}

with open('../data/trained_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Model successfully persisted.")